In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import re
import os
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.utils import find_project_root

BASE_DIR = find_project_root()

DATA_DIR = BASE_DIR / "data" 

from figures.fig_scripts.fig5_data_processing import *

In [3]:
plt.rc("pdf", fonttype=42)
plt.style.use("ggplot")
sns.set_style("whitegrid")

#### Import data from spatial reconstruction paper

In [4]:
# Sptaial data files:
umi_data_b = pd.read_csv(DATA_DIR / "external" / "table_B_scRNAseq_UMI_counts.tsv", sep="\t")
cell_zone_table_with_crypt = pd.read_csv(
    DATA_DIR / "external" / "cell_zone_table_with_crypt.txt", sep="\t"
)

# Landmark genes from Shalev paper:
lm_genes = pd.read_csv(DATA_DIR / "external" / "landmark_genes.txt", sep="\t", header=None)
lm_genes_upper = lm_genes[0].str.upper().tolist()

### Load raw data files:

In [5]:
raw_data, meta_data, features, barcodes = load_raw_data_files()

In [6]:
adata = generate_raw_adata(raw_data, meta_data, features, barcodes)

### Make Anndata object with baseline filtering:

In [7]:
adata_cell_filtered = prepare_cell_filtering(adata)

Cell filtering summary:
  Removed : 3846 cells
  Retained: 24765 cells
  Matrix after cells filtering: (24765, 27999)
  Cell matrix shape change:
  Shape before: (28611, 27999)
  Shape after : (24765, 27999)


/home/labs/antebilab/guyilan/.conda/envs/paper_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [8]:
adata_cell_filtered = remove_unknown_samples(adata_cell_filtered)


Unknown sample filtering summary:
  Removed : 4273 cells
  Retained: 20492 cells
  Matrix after cells filtering: (20492, 27999)
  Sample matrix shape change:
  Shape before: (24765, 27999)
  Shape after : (20492, 27999)


/home/labs/antebilab/guyilan/.conda/envs/paper_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [9]:
adata_cell_dup_filtered = remove_lower_duplicates(adata_cell_filtered)

Duplicate gene merging summary:
  Removed : 66 genes
  Retained: 27933 genes
  Matrix after genes filtering: (20492, 27933)
  Gene matrix shape change:
  Shape before: (20492, 27999)
  Shape after : (20492, 27933)


In [10]:
adata_cell_dup_filtered, h2bcit_irne_expression = seperate_h2b(
    adata_cell_dup_filtered, save=True
)

H2BCITRINE separation summary:
  Removed : 1 genes
  Retained: 27932 genes
  Matrix after genes filtering: (20492, 27932)
  Gene matrix shape change:
  Shape before: (20492, 27933)
  Shape after : (20492, 27932)
adata without H2BCITRINE saved.
H2BCITRINE expression saved.


### Generate final dataframe for training:

In [11]:
adata = sc.read_h5ad(DATA_DIR / "processed" / 'fig5_data' / "adata_cell_dup_filtered.h5ad")

In [12]:
adata_filtered = filter_data(adata, ligands="BMP4|BMP6|BMP9")

Filter to ligands: BMP4|BMP6|BMP9
Ligand filtering summary:
  Removed : 10005 cells
  Retained: 10487 cells
  Matrix after cells filtering: (10487, 27932)
  Ligand subset shape change:
  Shape before: (20492, 27932)
  Shape after : (10487, 27932)


In [13]:
n_top = 2250
ligands = ["BMP4", "BMP6", "BMP9", "CTRL"]
train_obj = process_df_for_final(
    adata_filtered,
    ligands,
    min_cell=20,
    n_top=n_top,
    log2=False,
    highly_variable_genes=True,
    rank_conc=True,
    save_before_hvg=True,
    attach_barcodes=True,
)

/home/labs/antebilab/guyilan/.conda/envs/paper_env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


Minimum-cell gene filtering summary:
  Removed : 14293 genes
  Retained: 13639 genes
  Matrix after genes filtering: (10487, 13639)
  Adata shape change:
  Shape before: (10487, 27932)
  Shape after : (10487, 13639)
Filtered adata to genes with at least 20 cells
Processing adata with dimensions: (10487, 13639)
Applying natural log transformation
Adata before HVG filtering saved.
Highly variable gene selection summary:
  Removed : 11389 genes
  Retained: 2250 genes
  Matrix after genes filtering: (10487, 2250)
  Adata shape change:
  Shape before: (10487, 13639)
  Shape after : (10487, 2250)


In [ ]:
train_obj_final = prepare_data_for_modeling(
    train_obj,
    normal_stratify=True,
    scale=False,
)

Encoded Labels: [0 2 0 ... 6 5 5]
Label Mapping:
0 -> 0
1 -> 1
2 -> 2
3 -> 3
4 -> 4
5 -> 5
6 -> 6
7 -> 7
Model feature preparation summary:
  Removed : 3 columns
  Retained: 2250 columns
  Matrix after columns filtering: (10487, 2250)
  Modeling dataframe shape change:
  Shape before: (10487, 2253)
  Shape after : (10487, 2250)
Splitting data into train and test sets
  Barcode vectors: train=8389, test=2098
  Train matrix shape: (8389, 2250)
  Test matrix shape : (2098, 2250)


In [ ]:
export_train(
    train_obj_final, output_folder=DATA_DIR / "processed" / 'fig5_data', prefix=f"_{n_top}"
)

Exported X_train -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/X_train_2250.pkl
Exported X_test -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/X_test_2250.pkl
Exported y_train -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/y_train_2250.npy
Exported y_test -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/y_test_2250.npy
Exported df_final -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/df_final_2250.pkl
Exported adata_final -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/adata_final_2250.h5ad
Exported barcode_train -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/barcode_train_2250.npy
Exported barcode_test -> /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/barcode_test_2250.npy


In [ ]:
spatial_obj = generate_spatial_data(train_obj_final, umi_data_b, cell_zone_table_with_crypt)

UMI preprocessing summary:
  Removed : 1 rows
  Retained: 1383 rows
  Matrix after rows filtering: (1383, 27998)
  UMI matrix shape change:
  Shape before: (27998, 1384)
  Shape after : (1383, 27998)
Spatial AnnData generation summary:
  Input matrix shape : (1383, 27998)
  Output AnnData shape: (1383, 27998)
Using natural log transformation for spatial data
Spatial feature alignment summary:
  Removed : 25748 genes
  Retained: 2250 genes
  Matrix after genes filtering: (1383, 2250)
  Spatial dataframe shape change:
  Shape before: (1383, 27998)
  Shape after : (1383, 2250)
  Spatial modeling matrix shape: (1383, 2250)


In [21]:
export_spatial(
    spatial_obj, output_folder=DATA_DIR / "processed" / 'fig5_data', prefix=f"_{n_top}"
)

Skipping df_spatial — /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/df_spatial_2250.pkl already exists.
Skipping adata_spatial — /home/labs/antebilab/guyilan/master/rec_paper/data/processed/fig5_data/adata_spatial_2250.h5ad already exists.
